<a href="https://colab.research.google.com/github/Vinicius-Mur/Homework-AI-Agent/blob/main/AI_English_Homework_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Instala os motores de renderização de alta fidelidade para o WeasyPrint
!apt-get install -y -qq libpango-1.0-0 libpangoft2-1.0-0 > /dev/null
!pip install -q weasyprint

In [2]:
# ==============================================================================
# CÉLULA 1: AMBIENTE, GOOGLE DRIVE E CONFIGURAÇÕES DO SISTEMA
# ==============================================================================
print("📦 1/4 Instalando dependências...")
!pip install -q -U "google-genai>=0.1.0" reportlab pydantic --no-warn-conflicts

import os
import sys
from google.colab import drive, userdata
from google import genai

print("📁 2/4 Montando o Google Drive...")
drive.mount('/content/drive', force_remount=True)

print("🔑 3/4 Autenticando Gemini...")
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=gemini_key)
    print("✅ Autenticado com sucesso!")
except Exception as e:
    print(f"⚠️ Erro de API: {e}")

# Definição dos caminhos no seu Google Drive
PROJECT_PATH = '/content/drive/MyDrive/AI_English_Teacher_Agent'
SAVE_PATH = os.path.join(PROJECT_PATH, 'Salvos')
SYSTEM_PATH = os.path.join(PROJECT_PATH, 'System') # <--- Nova pasta com os seus arquivos .py e .json

os.makedirs(SAVE_PATH, exist_ok=True)
os.makedirs(SYSTEM_PATH, exist_ok=True)

print("🎨 4/4 Mapeando arquivos de sistema do Drive...")

# Adiciona a pasta System ao caminho do Python para permitir importações de arquivos .py
if SYSTEM_PATH not in sys.path:
    sys.path.append(SYSTEM_PATH)

# Verifica se os arquivos base estão presentes no Drive
tokens_file = os.path.join(SYSTEM_PATH, 'design_tokens.json')
if os.path.exists(tokens_file):
    print("✅ design_tokens.json detectado no Google Drive!")
else:
    print(f"⚠️ ATENÇÃO: Faça o upload do 'design_tokens.json' para a pasta: {SYSTEM_PATH}")

# Baixar fontes padrão do Linux (DejaVu) para evitar erros de tipografia
!apt-get install -y -qq fonts-dejavu > /dev/null

print("✅ Ambiente configurado. Os seus arquivos estão seguros no Drive e prontos para uso.")

import os
import urllib.request

# Criar a pasta local onde o lessonkit.py procura as fontes
font_dir = '/content/fonts'
os.makedirs(font_dir, exist_ok=True)

font_urls = {
    'Carlito-Regular.ttf': 'https://github.com/google/fonts/raw/main/ofl/carlito/Carlito-Regular.ttf',
    'Carlito-Bold.ttf': 'https://github.com/google/fonts/raw/main/ofl/carlito/Carlito-Bold.ttf',
    'Carlito-Italic.ttf': 'https://github.com/google/fonts/raw/main/ofl/carlito/Carlito-Italic.ttf',
    'Carlito-BoldItalic.ttf': 'https://github.com/google/fonts/raw/main/ofl/carlito/Carlito-BoldItalic.ttf',
    'Poppins-Medium.ttf': 'https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-Medium.ttf',
    'Poppins-Bold.ttf': 'https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-Bold.ttf',
    'Poppins-MediumItalic.ttf': 'https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-MediumItalic.ttf',
    'Poppins-BoldItalic.ttf': 'https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-BoldItalic.ttf',
    'DejaVuSans.ttf': 'https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSans.ttf',
    'DejaVuSans-Bold.ttf': 'https://github.com/dejavu-fonts/dejavu-fonts/raw/master/ttf/DejaVuSans-Bold.ttf'
}

print("📦 Descarregando fontes tipográficas para /content/fonts...")
for font_name, url in font_urls.items():
    filepath = os.path.join(font_dir, font_name)
    if not os.path.exists(filepath):
        try:
            urllib.request.urlretrieve(url, filepath)
        except Exception as e:
            print(f"⚠️ Erro ao transferir {font_name}: {e}")

print("✅ Fontes prontas para o lessonkit.py!")

📦 1/4 Instalando dependências...
📁 2/4 Montando o Google Drive...
Mounted at /content/drive
🔑 3/4 Autenticando Gemini...
✅ Autenticado com sucesso!
🎨 4/4 Mapeando arquivos de sistema do Drive...
✅ design_tokens.json detectado no Google Drive!
✅ Ambiente configurado. Os seus arquivos estão seguros no Drive e prontos para uso.
📦 Descarregando fontes tipográficas para /content/fonts...
⚠️ Erro ao transferir DejaVuSans.ttf: HTTP Error 404: Not Found
⚠️ Erro ao transferir DejaVuSans-Bold.ttf: HTTP Error 404: Not Found
✅ Fontes prontas para o lessonkit.py!


In [3]:
# ==============================================================================
# CÉLULA 2: SISTEMA RAG - LEITURA INTELIGENTE DE CONTEXTO DO DRIVE
# ==============================================================================
import os

def extract_rag_context(target_level: str) -> dict:
    """
    Busca referências contextuais navegando nas pastas do Drive (A1 ou A2)
    e categoriza a leitura (Gramática, Idioms, Exercícios, Escrita).
    """
    base_folder = os.path.join(globals().get("PROJECT_PATH", ""), target_level)
    context_data = {"grammar": "", "idioms": "", "exercises": "", "writing": ""}

    if not os.path.exists(base_folder):
        print(f"⚠️ Pasta {base_folder} não encontrada. Usando RAG simulado.")
        return context_data

    # Mapeamento do RAG de acordo com as pastas do utilizador
    rag_map = {
        "grammar": ["Student Book"],
        "idioms": ["Student Book Listening Scripts", "Videos Scripts", "Sudent Book Script"],
        "exercises": ["Workbook", "Units Review", "Unit Reviews"],
        "writing": ["Writing Practice"]
    }

    # Varre a pasta A1 ou A2
    for subfolder in os.listdir(base_folder):
        sub_path = os.path.join(base_folder, subfolder)
        if os.path.isdir(sub_path):
            # Identifica a qual categoria esta pasta pertence
            category = None
            for cat, folder_names in rag_map.items():
                if subfolder in folder_names:
                    category = cat
                    break

            if category:
                # Lê os .txt dentro desta subpasta
                for file in os.listdir(sub_path):
                    if file.endswith('.txt'):
                        try:
                            with open(os.path.join(sub_path, file), 'r', encoding='utf-8') as f:
                                context_data[category] += f.read() + "\n"
                        except Exception as e:
                            pass # Ignora ficheiros corrompidos

    return context_data

In [4]:
# ==============================================================================
# CÉLULA 3: MOTOR PEDAGÓGICO (GEMINI + FALLBACK + PYDANTIC)
# ==============================================================================
import time
import random
from typing import List
from pydantic import BaseModel, Field
from google import genai
from google.genai import errors
from google.colab import userdata

# 1. Schemas avançados para o PDF profissional
class ShadowingPractice(BaseModel):
    sentence: str = Field(description="Frase-chave exata")
    tips: str = Field(description="Dicas de stress e entonação")

class WordBankItem(BaseModel):
    word: str = Field(description="Palavra ou Lexical Chunk")
    meaning_or_false_friend: str = Field(description="Significado breve, contexto ou alerta de falso cognato")

class TBLTMission(BaseModel):
    title: str = Field(description="Título da tarefa prática")
    role_problem: str = Field(description="Roleplay e o problema real a resolver")
    steps: List[str] = Field(description="Passos de execução")

class ModernHomeworkSchema(BaseModel):
    lesson_title: str = Field(description="Título da lição")
    cefr_level: str = Field(description="A1 ou A2")
    focus_summary: str = Field(description="Resumo do foco gramatical/lexical")
    teacher_lead_in: str = Field(description="Notas do professor (PT-BR) para a introdução")
    comprehensible_input_text: str = Field(description="Texto/Diálogo rico (i+1)")
    word_bank: List[WordBankItem] = Field(description="Vocabulário e chunks principais extraídos do input")
    shadowing_guide: List[ShadowingPractice] = Field(description="Guia de Shadowing do texto")
    tblt_mission: TBLTMission = Field(description="A Missão final (TBLT)")

# 2. Lógica de Fallback e Rotação de Modelos
MODEL_PREFERENCE = ["gemini-3.8-flash"]
RETRYABLE = {429, 500, 502, 503, 504}

def _usable_models(client):
    try:
        live = {m.name.replace("models/", "") for m in client.models.list()
                if "generateContent" in (getattr(m, "supported_actions", None) or [])}
        return [m for m in MODEL_PREFERENCE if m in live] or MODEL_PREFERENCE
    except Exception:
        return MODEL_PREFERENCE

def generate_with_fallback(client, prompt, schema, attempts_per_model=3):
    last_error = None
    for model in _usable_models(client):
        for attempt in range(1, attempts_per_model + 1):
            try:
                print(f"🧠 {model} (tentativa {attempt}/{attempts_per_model})...")
                response = client.models.generate_content(
                    model=model,
                    contents=prompt,
                    config={
                        "response_mime_type": "application/json",
                        "response_schema": schema,
                        "temperature": 0.2,
                    },
                )
                return response.text
            except errors.APIError as e:
                last_error = e
                if e.code not in RETRYABLE:
                    raise  # Erro real (chave, schema, etc.): não adianta repetir
                wait = min(2 ** attempt, 30) + random.uniform(0, 1)
                print(f"   ⏳ Erro {e.code}. Nova tentativa em {wait:.0f}s...")
                time.sleep(wait)
        print(f"   ↪ {model} indisponível, trocando de modelo...")
    raise RuntimeError(f"Todos os modelos falharam. Último erro: {last_error}")

# 3. Função Principal
def generate_modern_homework(target_level: str, grammar_point: str, vocab_list: List[str]) -> str:
    try:
        active_client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    except Exception as e:
        raise ValueError("Erro ao carregar API Key.")

    # Busca o contexto RAG nas pastas (função definida na Célula 2)
    rag_context = extract_rag_context(target_level)

    prompt = f"""
Design a highly professional Modern TBLT Homework Worksheet for CEFR Level {target_level}.

INPUTS:
Level: {target_level}
Grammar Focus: {grammar_point}
Lexical Concepts: {', '.join(vocab_list)}

RAG CONTEXT TO USE (If available):
Grammar Style: {rag_context['grammar'][:1000]}
Idioms/Expressions: {rag_context['idioms'][:1000]}
Task/Exercise Inspiration: {rag_context['exercises'][:1000]}
Writing Model: {rag_context['writing'][:1000]}

Follow the exact JSON schema. Generate a compelling comprehensible input text. NO passive grammar drills.
"""
    return generate_with_fallback(active_client, prompt, ModernHomeworkSchema)

In [5]:
# ==============================================================================
# CÉLULA 4: MOTOR DE RENDERIZAÇÃO REPORTLAB (DESIGN SYSTEM AVANÇADO)
# ==============================================================================
# ==============================================================================
# CÉLULA 4: MOTOR DE RENDERIZAÇÃO WEASYPRINT (DESIGN SYSTEM AVANÇADO)
# ==============================================================================
import os
from weasyprint import HTML

def build_advanced_pdf(json_data: dict, output_path: str):
    print("📄 Construindo PDF com Design Avançado (WeasyPrint)...")

    # Função segura para evitar erros caso algum campo venha vazio
    def get_text(key, default=""): return json_data.get(key, default) or default

    # Construção do HTML com o CSS do seu Design System
    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@500;700&family=Carlito:ital,wght@0,400;0,700;1,400&display=swap" rel="stylesheet">
        <style>
            @page {{
                size: A4;
                margin: 24mm 18mm 20mm 18mm;
            }}
            body {{
                font-family: 'Carlito', sans-serif;
                color: #1E2A38;
                font-size: 11.5pt;
                line-height: 1.5;
                margin: 0;
            }}
            h1, h2, h3, .poppins {{ font-family: 'Poppins', sans-serif; margin: 0; }}

            /* Cover Band (Barra do Topo) */
            .cover-band {{
                background-color: #1E2A38;
                color: #FFFFFF;
                padding: 18px 24px;
                border-radius: 8px 8px 0 0;
                margin-bottom: 15px;
            }}
            .cover-band h1 {{ font-size: 22pt; line-height: 1.1; text-transform: uppercase; letter-spacing: 0.5px; }}
            .cover-band p {{ font-size: 11pt; color: #D9DEE5; margin-top: 8px; font-family: 'Carlito', sans-serif; margin-bottom: 0; }}

            /* Teacher Rail (Caixa de Notas) */
            .teacher-rail {{
                background-color: #F3F5F8;
                border-left: 5px solid #5B6776;
                padding: 12px 18px;
                font-size: 10.5pt;
                color: #5B6776;
                margin-bottom: 25px;
                border-radius: 0 6px 6px 0;
            }}
            .teacher-rail strong {{ font-family: 'Poppins', sans-serif; font-size: 9pt; color: #1E2A38; letter-spacing: 1px; }}

            /* Cabeçalhos de Seção com Badges */
            .section-title {{
                font-size: 14pt;
                color: #1E2A38;
                margin-bottom: 12px;
                text-transform: uppercase;
                font-family: 'Poppins', sans-serif;
            }}
            .badge {{
                color: white;
                padding: 3px 10px;
                border-radius: 6px;
                font-size: 12pt;
                margin-right: 8px;
                vertical-align: middle;
            }}
            .card {{ padding: 18px; border-radius: 8px; margin-bottom: 24px; }}

            /* Cores Semânticas do seu Design System */
            .badge-blue {{ background-color: #286CB0; }}
            .card-blue {{ background-color: #E8F1FB; border-left: 6px solid #286CB0; }}

            .badge-yellow {{ background-color: #F2B600; }}
            .card-yellow {{ background-color: #FFF6D6; border-left: 6px solid #F2B600; }}

            .badge-green {{ background-color: #2F9E6B; }}
            .card-green {{ background-color: #E4F5EC; border-left: 6px solid #2F9E6B; }}

            .badge-dark {{ background-color: #1E2A38; }}
            .card-dark {{ border: 2px solid #1E2A38; padding: 0; border-radius: 8px; margin-bottom: 24px; overflow: hidden; }}

            /* Ajustes Internos */
            .word-item {{ margin-bottom: 8px; }}
            .shadow-item {{ margin-bottom: 14px; }}
            .shadow-tip {{ font-style: italic; color: #5B6776; font-size: 10.5pt; display: block; margin-top: 2px; }}

            /* TBLT Mission Bar */
            .mission-bar {{ background-color: #1E2A38; color: white; padding: 12px 18px; font-family: 'Poppins', sans-serif; font-size: 11.5pt; letter-spacing: 0.5px; }}
            .mission-content {{ padding: 18px; }}
        </style>
    </head>
    <body>
        <!-- Cabeçalho -->
        <div class="cover-band">
            <h1>{get_text('lesson_title', 'Lesson Pack')}</h1>
            <p><strong>LEVEL:</strong> {get_text('cefr_level', 'A2')} &nbsp;|&nbsp; <strong>FOCUS:</strong> {get_text('focus_summary')}</p>
        </div>

        <!-- Notas do Professor -->
        <div class="teacher-rail">
            <strong>TEACHER NOTES</strong><br>
            {get_text('teacher_lead_in')}
        </div>

        <!-- 01 Input (Azul) -->
        <div class="section-title"><span class="badge badge-blue">01</span> COMPREHENSIBLE INPUT</div>
        <div class="card card-blue">
            {get_text('comprehensible_input_text').replace(chr(10), '<br>')}
        </div>
    """

    # Bloco Amarelo (Vocabulário)
    wb_list = json_data.get('word_bank', [])
    if wb_list:
        html_content += '<div class="section-title"><span class="badge badge-yellow">VOCAB</span> WORD BANK</div><div class="card card-yellow">'
        for wb in wb_list:
            html_content += f"<div class='word-item'><strong>{wb.get('word')}</strong>: {wb.get('meaning_or_false_friend')}</div>"
        html_content += '</div>'

    # Bloco Verde (Shadowing)
    sh_list = json_data.get('shadowing_guide', [])
    if sh_list:
        html_content += '<div class="section-title"><span class="badge badge-green">02</span> SHADOWING</div><div class="card card-green">'
        for idx, sh in enumerate(sh_list, 1):
            html_content += f"<div class='shadow-item'><strong>{idx}. {sh.get('sentence')}</strong><span class='shadow-tip'>Tip: {sh.get('tips')}</span></div>"
        html_content += '</div>'

    # Bloco Escuro (Missão TBLT)
    mission = json_data.get('tblt_mission', {})
    if mission:
        html_content += f"""
        <div class="section-title"><span class="badge badge-dark">03</span> TASK-BASED LEARNING</div>
        <div class="card card-dark">
            <div class="mission-bar">MISSION: {mission.get('title', '').upper()}</div>
            <div class="mission-content">
                <p><strong>Role & Problem:</strong> {mission.get('role_problem', '')}</p>
                <ul style="padding-left: 20px; margin-bottom: 0;">
        """
        for step in mission.get('steps', []):
            html_content += f"<li style='margin-bottom: 8px;'>{step}</li>"
        html_content += "</ul></div></div>"

    html_content += """
    </body>
    </html>
    """

    # Gerar o PDF
    HTML(string=html_content).write_pdf(output_path)
    print(f"✨ PDF Editorial guardado em: {output_path}")

In [6]:

# ==============================================================================
# CÉLULA 5: AGENTE INTERATIVO UI (COM CHECKPOINT JSON)
# ==============================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import os

class EnglishTeacherAgentUI:
    def __init__(self):
        self.header = widgets.HTML(value="<h3>🤖 AI English Teacher Agent (Pro + Fallback)</h3><p>Geração blindada contra falhas de rede (salva JSON automático).</p>")
        self.level_dropdown = widgets.Dropdown(options=['A1', 'A2'], value='A2', description='Nível:')
        self.grammar_input = widgets.Text(value='Present Continuous', description='Gramática:', layout=widgets.Layout(width='80%'))
        self.vocab_input = widgets.Textarea(value='Travel, Airport, Baggage', description='Vocabulário:', layout=widgets.Layout(width='80%', height='60px'))
        self.generate_btn = widgets.Button(description='Gerar Lesson Pack', button_style='primary', layout=widgets.Layout(width='40%'))
        self.generate_btn.on_click(self._on_generate)
        self.output_log = widgets.Output()
        display(widgets.VBox([self.header, self.level_dropdown, self.grammar_input, self.vocab_input, self.generate_btn, self.output_log]))

    def _on_generate(self, b):
        with self.output_log:
            clear_output()
            level = self.level_dropdown.value
            grammar = self.grammar_input.value
            vocab = [v.strip() for v in self.vocab_input.value.split(',')]

            try:
                # 1. Raciocínio LLM (Agora com Fallback automático)
                json_str = generate_modern_homework(level, grammar, vocab)
                lesson_data = json.loads(json_str)

                safe_title = grammar.replace(" ", "_")[:15]

                # 2. CHECKPOINT: Salva o JSON no Drive para evitar perda de dados
                json_path = os.path.join(SAVE_PATH, f"Lesson_{level}_{safe_title}.json")
                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(lesson_data, f, ensure_ascii=False, indent=2)
                print(f"💾 Checkpoint JSON salvo em: {json_path}")

                # 3. Renderização PDF Profissional (Função da Célula 4)
                output_pdf = os.path.join(SAVE_PATH, f"Pro_Lesson_{level}_{safe_title}.pdf")
                build_advanced_pdf(lesson_data, output_pdf)

            except Exception as e:
                print(f"\n❌ Erro durante a execução: {e}")

EnglishTeacherAgentUI()